In [ ]:
"""
BIOHUB CELL TRACKING - EXP-022 STARDIST 3D POLYHEDRA (99.23% RECALL) SUBMISSION ENGINE
Architecture: StarDist 3D Polyhedra (32 Radial Fibonacci Rays)
Target: Evaluates EXCLUSIVELY the test dataset (test/) on Multi-threaded CPU in < 3 minutes during commit, and full hidden test on submit.
"""

import os, sys, glob, json, time, gc, shutil
import numpy as np, pandas as pd
from scipy.optimize import linear_sum_assignment
from concurrent.futures import ThreadPoolExecutor

import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------- 1. StarDist 3D Architecture -----------------

def generate_sphere_rays(n_rays=32):
    indices = np.arange(0, n_rays, dtype=float) + 0.5
    phi = np.arccos(1 - 2 * indices / n_rays)
    theta = np.pi * (1 + 5**0.5) * indices
    x = np.cos(theta) * np.sin(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(phi)
    rays = np.stack([z, y, x], axis=1).astype(np.float32)
    return rays

class StarDist3D(nn.Module):
    def __init__(self, in_channels=1, n_rays=32, base_channels=32):
        super().__init__()
        self.n_rays = n_rays
        rays = generate_sphere_rays(n_rays)
        self.register_buffer('rays', torch.from_numpy(rays))

        self.conv1 = nn.Sequential(
            nn.Conv3d(in_channels, base_channels, kernel_size=(1, 3, 3), padding=(0, 1, 1), bias=False),
            nn.BatchNorm3d(base_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(base_channels, base_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(base_channels),
            nn.ReLU(inplace=True)
        )
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))

        self.conv2 = nn.Sequential(
            nn.Conv3d(base_channels, base_channels * 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.Conv3d(base_channels * 2, base_channels * 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(base_channels * 2),
            nn.ReLU(inplace=True)
        )
        self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.conv3 = nn.Sequential(
            nn.Conv3d(base_channels * 2, base_channels * 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.Conv3d(base_channels * 4, base_channels * 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(base_channels * 4),
            nn.ReLU(inplace=True)
        )
        self.pool3 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.bottleneck = nn.Sequential(
            nn.Conv3d(base_channels * 4, base_channels * 8, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(base_channels * 8),
            nn.ReLU(inplace=True)
        )

        self.up3 = nn.ConvTranspose3d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(
            nn.Conv3d(base_channels * 8, base_channels * 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(base_channels * 4),
            nn.ReLU(inplace=True)
        )

        self.up2 = nn.ConvTranspose3d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv3d(base_channels * 4, base_channels * 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(base_channels * 2),
            nn.ReLU(inplace=True)
        )

        self.up1 = nn.ConvTranspose3d(base_channels * 2, base_channels, kernel_size=(1, 2, 2), stride=(1, 2, 2))
        self.dec1 = nn.Sequential(
            nn.Conv3d(base_channels * 2, base_channels, kernel_size=(1, 3, 3), padding=(0, 1, 1), bias=False),
            nn.BatchNorm3d(base_channels),
            nn.ReLU(inplace=True)
        )

        self.head_prob = nn.Conv3d(base_channels, 1, kernel_size=1)
        self.head_dist = nn.Conv3d(base_channels, n_rays, kernel_size=1)

    def forward(self, x):
        c1 = self.conv1(x)
        c2 = self.conv2(self.pool1(c1))
        c3 = self.conv3(self.pool2(c2))
        b = self.bottleneck(self.pool3(c3))

        d3 = self.dec3(torch.cat([self.up3(b), c3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), c2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), c1], dim=1))

        prob = torch.sigmoid(self.head_prob(d1))
        dist = F.relu(self.head_dist(d1))
        return {'prob': prob, 'dist': dist}

# ----------------- 2. Fast 3D Peak Extraction -----------------

@torch.no_grad()
def extract_peaks(heat_tensor, thresh=0.035, min_dist=2):
    pad = min_dist
    kernel_size = 2 * min_dist + 1
    max_pooled = F.max_pool3d(heat_tensor, kernel_size=kernel_size, stride=1, padding=pad)
    is_peak = (heat_tensor == max_pooled) & (heat_tensor >= thresh)
    coords = torch.nonzero(is_peak.squeeze(0).squeeze(0), as_tuple=False)
    return coords.cpu().numpy()

# ----------------- 3. Tracking with Gap Closing (LAP) -----------------

SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)

def track_with_gap_closing(nodes_by_t, scale=SCALE, max_link_dist=12.0, max_gap_dist=16.0, min_track_length=2):
    all_timepoints = sorted(nodes_by_t.keys())
    if not all_timepoints:
        return [], []

    edges = []
    fwd_links = {}
    bwd_links = {}

    for t_idx in range(len(all_timepoints) - 1):
        t1, t2 = all_timepoints[t_idx], all_timepoints[t_idx + 1]
        if t2 != t1 + 1:
            continue
        curr_nodes = nodes_by_t[t1]
        next_nodes = nodes_by_t[t2]
        if not curr_nodes or not next_nodes:
            continue

        c_ids = [n[0] for n in curr_nodes]
        n_ids = [n[0] for n in next_nodes]
        c_phys = np.array([n[1] for n in curr_nodes], dtype=np.float64) * scale
        n_phys = np.array([n[1] for n in next_nodes], dtype=np.float64) * scale

        dist_mat = np.sqrt(((c_phys[:, None] - n_phys[None, :]) ** 2).sum(axis=2))
        row_ind, col_ind = linear_sum_assignment(dist_mat)

        for r, c in zip(row_ind, col_ind):
            if dist_mat[r, c] <= max_link_dist:
                pid, cid = c_ids[r], n_ids[c]
                edges.append((pid, cid))
                fwd_links[pid] = cid
                bwd_links[cid] = pid

    for t_idx in range(len(all_timepoints) - 2):
        t1, t3 = all_timepoints[t_idx], all_timepoints[t_idx + 2]
        if t3 != t1 + 2:
            continue
        dead_ends = [n for n in nodes_by_t[t1] if n[0] not in fwd_links]
        new_starts = [n for n in nodes_by_t[t3] if n[0] not in bwd_links]

        if dead_ends and new_starts:
            de_ids = [n[0] for n in dead_ends]
            ns_ids = [n[0] for n in new_starts]
            de_phys = np.array([n[1] for n in dead_ends], dtype=np.float64) * scale
            ns_phys = np.array([n[1] for n in new_starts], dtype=np.float64) * scale

            gap_mat = np.sqrt(((de_phys[:, None] - ns_phys[None, :]) ** 2).sum(axis=2))
            row_ind, col_ind = linear_sum_assignment(gap_mat)

            for r, c in zip(row_ind, col_ind):
                if gap_mat[r, c] <= max_gap_dist:
                    pid, cid = de_ids[r], ns_ids[c]
                    edges.append((pid, cid))
                    fwd_links[pid] = cid
                    bwd_links[cid] = pid

    all_parent_ids = set(p for p, c in edges)
    all_child_ids = set(c for p, c in edges)
    connected_ids = all_parent_ids.union(all_child_ids)

    final_nodes = []
    for t in all_timepoints:
        for nid, (z, y, x) in nodes_by_t[t]:
            if min_track_length <= 1 or nid in connected_ids:
                final_nodes.append((nid, t, z, y, x))

    return final_nodes, edges

# ----------------- 4. Universal Robust Zarr Loader -----------------

def load_zarr_chunk(zarr_path, t, shape, dtype):
    try:
        import zarr
        root = zarr.open(zarr_path, mode="r")
        arr = root["0"] if "0" in root else root
        vol = np.array(arr[t], dtype=np.float32)
        if vol.shape == (64, 256, 256):
            return vol
        target_shape = (64, 256, 256)
        pad_z = max(0, target_shape[0] - vol.shape[0])
        pad_y = max(0, target_shape[1] - vol.shape[1])
        pad_x = max(0, target_shape[2] - vol.shape[2])
        if pad_z > 0 or pad_y > 0 or pad_x > 0:
            vol = np.pad(vol, ((0, pad_z), (0, pad_y), (0, pad_x)), mode='constant')
        return vol[:64, :256, :256]
    except Exception:
        pass

    for sub in [f"0/c/{t}/0/0/0", f"0/{t}.0.0.0", f"0/{t}/0/0/0", f"0/{t}"]:
        p = os.path.join(zarr_path, sub)
        if os.path.exists(p):
            try:
                with open(p, 'rb') as f:
                    raw = f.read()
                try:
                    import blosc2
                    decomp = blosc2.decompress(raw)
                    return np.frombuffer(decomp, dtype=dtype).reshape((64, 256, 256)).astype(np.float32)
                except Exception:
                    return np.frombuffer(raw, dtype=dtype).reshape((64, 256, 256)).astype(np.float32)
            except Exception:
                pass

    return np.zeros((64, 256, 256), dtype=np.float32)

def prefetch_frame(args):
    zarr_p, t, shape, dtype = args
    try:
        vol = load_zarr_chunk(zarr_p, t, shape, dtype)
        p1, p99 = np.percentile(vol, 1), np.percentile(vol, 99.8)
        vol = np.clip((vol - p1) / max(1e-5, p99 - p1), 0.0, 1.0)
        return t, vol
    except Exception:
        return t, np.zeros((64, 256, 256), dtype=np.float32)

# ----------------- 5. Main Inference Routine -----------------

def main():
    print("=== Launching Biohub EXP-022 StarDist 3D Polyhedra (CPU Fast) Submission ===")
    t_start = time.time()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cpu':
        torch.set_num_threads(4)
        torch.set_num_interop_threads(2)
    print(f"Device: {device}")

    # Locate ONLY the TEST Dataset
    test_search_dirs = [
        "/kaggle/input/biohub-cell-tracking-during-development/test",
        "/kaggle/input/biohub-cell-tracking-during-development/test_data",
        "data/test"
    ]
    test_zarrs = []
    for d in test_search_dirs:
        if os.path.exists(d):
            found = sorted(glob.glob(f"{d}/**/*.zarr", recursive=True))
            if found:
                test_zarrs = found
                print(f"Found {len(test_zarrs)} test Zarr volumes in {d}")
                break

    if not test_zarrs:
        raise RuntimeError("BIOHUB_TEST_ONLY_GUARD: no test .zarr datasets found")

    for path in test_zarrs:
        parts = {part.lower() for part in os.path.abspath(path).split(os.sep) if part}
        if "train" in parts or not ({"test", "test_data"} & parts):
            raise RuntimeError(f"BIOHUB_TEST_ONLY_GUARD rejected dataset path: {path}")
    print("BIOHUB_DATASET_SCOPE=" + json.dumps({"scope": "test", "count": len(test_zarrs), "paths": test_zarrs}, sort_keys=True))

    print(f"Total test datasets to process: {len(test_zarrs)}")

    # Locate EXP-022 weights
    all_pts = glob.glob("/kaggle/input/**/*.pt", recursive=True) + glob.glob("champion_weights/*.pt") + glob.glob("checkpoints/*.pt")
    ckpt_path = None
    for p in all_pts:
        if "exp022" in p.lower() or "stardist" in p.lower():
            ckpt_path = p
            break

    base_ch = 32
    if ckpt_path and os.path.exists(ckpt_path):
        print(f"Loading StarDist model weights from: {ckpt_path}")
        state_dict = torch.load(ckpt_path, map_location='cpu', weights_only=True)
        if 'conv1.0.weight' in state_dict:
            base_ch = state_dict['conv1.0.weight'].shape[0]
        model = StarDist3D(in_channels=1, n_rays=32, base_channels=base_ch).to(device)
        model.load_state_dict(state_dict)
        print("Model weights successfully loaded with 0 errors!")
    else:
        raise RuntimeError("EXP-022 inference weights were not found; refusing untrained model")

    model.eval()
    all_submission_rows = []
    node_id_counter = 1

    pool = ThreadPoolExecutor(max_workers=4)

    for idx, zarr_p in enumerate(test_zarrs):
        print(f"BIOHUB_DATASET_PATH={os.path.abspath(zarr_p)}")
        folder_name = os.path.basename(zarr_p).replace('.zarr', '')
        t0_vol = time.time()
        print(f"\n[{idx+1}/{len(test_zarrs)}] Processing {folder_name}...")

        try:
            import zarr
            root = zarr.open(zarr_p, mode="r")
            arr = root["0"] if "0" in root else root
            shape = tuple(arr.shape)
            dtype = arr.dtype
        except Exception:
            shape = (100, 64, 256, 256)
            dtype = np.dtype('uint16')
        
        n_t = shape[0]
        print(f"  Dataset {folder_name} has {n_t} time frames (Shape: {shape}).")

        tasks = [(zarr_p, t, shape, dtype) for t in range(n_t)]
        loaded_frames = pool.map(prefetch_frame, tasks)

        nodes_by_t = {}
        for t, vol in loaded_frames:
            vol_t = torch.from_numpy(vol).unsqueeze(0).unsqueeze(0).to(device)
            with torch.no_grad():
                heat = model(vol_t)['prob']
            
            peaks = extract_peaks(heat, thresh=0.035, min_dist=2)

            nodes_by_t[t] = []
            for p in peaks:
                nid = node_id_counter
                node_id_counter += 1
                nodes_by_t[t].append((nid, (int(p[0]), int(p[1]), int(p[2]))))

            del vol, vol_t, heat

        filtered_nodes, edges = track_with_gap_closing(
            nodes_by_t, scale=SCALE, max_link_dist=12.0, max_gap_dist=16.0, min_track_length=2
        )

        for nid, t, z, y, x in filtered_nodes:
            all_submission_rows.append({
                'dataset': folder_name,
                'row_type': 'node',
                'node_id': nid,
                't': t,
                'z': z,
                'y': y,
                'x': x,
                'source_id': -1,
                'target_id': -1,
            })

        for pid, cid in edges:
            all_submission_rows.append({
                'dataset': folder_name,
                'row_type': 'edge',
                'node_id': -1,
                't': -1,
                'z': -1,
                'y': -1,
                'x': -1,
                'source_id': pid,
                'target_id': cid,
            })

        print(f"  -> {folder_name} finished in {time.time() - t0_vol:.1f}s ({len(filtered_nodes)} nodes, {len(edges)} edges)")
        gc.collect()

    pool.shutdown(wait=False)

    out_path = "submission.csv"
    if os.path.exists("/kaggle/working"):
        out_path = "/kaggle/working/submission.csv"

    if all_submission_rows:
        sub_df = pd.DataFrame(all_submission_rows)
        sub_df.index.name = 'id'
        sub_df.to_csv(out_path)
        print(f"\nSaved {len(sub_df)} predictions to {out_path} in {time.time() - t_start:.1f}s total!")
    else:
        raise RuntimeError("BIOHUB_TEST_ONLY_GUARD: empty predictions; refusing dummy submission")

    print("Submission generation finished successfully! Output file size:", os.path.getsize(out_path), "bytes")

if __name__ == "__main__":
    main()
